# 00 · Preparación de C2: manifiesto, caché, anotaciones y congelación

Deja C2 listo para los experimentos (§8.3 de la memoria; §3, §11, §13 y §15 de `experimentos.md`). No usa
LLM ni API.

| § | Qué hace | Escribe en |
|---|---|---|
| 2 | **Manifiesto de C2:** MD5, dimensiones, orientación EXIF y columnas manuales de origen y licencia | `experimentos/manifiesto_C2.csv` (versionado) |
| 3 | **Caché de percepción:** YOLO + saliencia + máscara del sujeto con el código real, en un proceso aparte | `outputs/experimentos/cache/percepcion/` |
| 4 | **Anotaciones:** plantillas A1 y A2, migración de las revisiones existentes, vigencia, hojas de revisión y validación | `experimentos/anotaciones/`, `outputs/experimentos/hojas_revision/` |
| 5 | **Tablas descriptivas de §8.3** | `experimentos/resultados/00/` |
| 6 | **Propuesta de R** (necesita A1) | nada: imprime el bloque YAML para pegarlo en la configuración |
| 7 | **Congelación** (solo con `CONGELAR = True`) | `experimentos/congelacion.json` |

### Cuándo ejecutarlo
1. **Ahora:** genera el manifiesto, la caché, las plantillas y las hojas.
2. **Tras anotar** en `anotaciones/ANOTAR_AQUI.xlsx` y pasarlo a los CSV con
   `experimentos/importar_anotaciones.py`: vuelve a ejecutarlo para validar las anotaciones y obtener las
   tablas de §8.3 y la propuesta de R.
3. **Tras hacer commit:** pon `CONGELAR = True` en §7.

**Nunca sobrescribe una anotación ni el origen y la licencia del manifiesto:** solo crea los ficheros que
faltan y añade columnas.

## 1. Configuración

In [ ]:
import sys, json, textwrap
from datetime import datetime
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ / "experimentos") not in sys.path:
    sys.path.insert(0, str(RAIZ / "experimentos"))
import comun
comun.preparar_sys_path()

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown, Image as ImagenIPy

from tfg_multiagente_fotografia.tools import espacio_aislamiento_tool as ea    # <- EL CÓDIGO REAL (solo funciones puras)
from tfg_multiagente_fotografia.tools.imagen import cargar_imagen

CFG = comun.cargar_config()
ANOT = comun.ruta_config("anotaciones")
RES = comun.dir_resultados("00")
HOJAS = RAIZ / "outputs" / "experimentos" / "hojas_revision"
pd.set_option("display.max_colwidth", 90)

C2 = comun.imagenes_C2()
STEMS = [p.stem for p in C2]
RUTA = dict(zip(STEMS, C2))
assert STEMS == [f"imagen{i}" for i in range(1, len(C2) + 1)], "La numeración de C2 no es imagen1..imagenN"
assert len(C2) == 100, f"C2 tiene {len(C2)} imágenes"
print(f"C2: {len(C2)} imágenes en {comun.relativa(C2[0].parent)}")

## 2. Manifiesto de C2

Una fila por imagen con lo que se puede medir (MD5, bytes, dimensiones, orientación, formato y EXIF de
fecha y cámara) y con cinco columnas **manuales**: `origen`, `autor`, `licencia`, `url` y `notas`. §8.3 las
necesita y hoy están [pendiente].

Al regenerarlo se conservan las columnas manuales. Si el MD5 de una imagen cambia, se avisa en `notas`:
el origen anotado pertenece a la imagen anterior.

Las dimensiones son las que ve OpenCV (`cv2.imread` aplica la orientación EXIF), que son las que usan los
especialistas. El clasificador abre con PIL, que **no** la aplica. Por eso una imagen con orientación EXIF
distinta de 1 se ve girada en el clasificador y derecha en los especialistas. Se listan aparte.

In [ ]:
RUTA_MANIFIESTO = RAIZ / "experimentos" / "manifiesto_C2.csv"
MANUALES = ["origen", "autor", "licencia", "url", "notas"]


def fila_manifiesto(p):
    with Image.open(p) as im:
        ancho, alto = im.size
        formato, modo = im.format, im.mode
        exif = im.getexif()
        orientacion_exif = int(exif.get(274, 1))
        fecha = exif.get_ifd(0x8769).get(36867) or exif.get(306) or ""
        camara = " ".join(str(exif.get(t, "")).strip() for t in (271, 272)).strip()
    if orientacion_exif in (5, 6, 7, 8):          # giro de 90°: OpenCV intercambia ancho y alto
        ancho, alto = alto, ancho
    return {
        "imagen": p.stem, "fichero": comun.relativa(p), "md5": comun.md5(p), "bytes": p.stat().st_size,
        "ancho": ancho, "alto": alto, "megapixeles": round(ancho * alto / 1e6, 2),
        "orientacion": "horizontal" if ancho > alto else "vertical" if alto > ancho else "cuadrada",
        "formato": formato, "modo": modo, "exif_orientacion": orientacion_exif,
        "fecha_captura": str(fecha), "camara": camara,
    }


auto = pd.DataFrame([fila_manifiesto(p) for p in C2]).set_index("imagen")
manual = pd.DataFrame("", index=auto.index, columns=MANUALES)
cambiadas = []
if RUTA_MANIFIESTO.exists():
    previo = comun.leer_csv_robusto(RUTA_MANIFIESTO).set_index("imagen")
    for col in MANUALES:
        if col in previo:
            manual[col] = previo[col].reindex(auto.index).fillna("")
    comunes = auto.index.intersection(previo.index)
    cambiadas = [s for s in comunes if previo.at[s, "md5"] != auto.at[s, "md5"]]
    for s in cambiadas:
        aviso = f"REVISAR origen: la imagen cambió (MD5 anterior {previo.at[s, 'md5']})"
        if aviso not in manual.at[s, "notas"]:
            manual.at[s, "notas"] = f"{aviso}. {manual.at[s, 'notas']}".strip(". ")

MANIFIESTO = auto.join(manual)
MANIFIESTO.reset_index().to_csv(RUTA_MANIFIESTO, index=False, encoding="utf-8")

duplicados = MANIFIESTO[MANIFIESTO.md5.duplicated(keep=False)]
assert duplicados.empty, f"Imágenes duplicadas en C2:\n{duplicados[['md5']]}"
girada_exif = MANIFIESTO.index[MANIFIESTO.exif_orientacion != 1].tolist()
sin_origen = MANIFIESTO.index[(MANIFIESTO.origen == "") | (MANIFIESTO.licencia == "")].tolist()

print(f"Manifiesto: {comun.relativa(RUTA_MANIFIESTO)} · {len(MANIFIESTO)} imágenes, todas con MD5 distinto")
print(f"Imágenes cambiadas desde el manifiesto anterior: {cambiadas or 'ninguna'}")
print(f"Orientación EXIF distinta de 1 (el clasificador las ve giradas): {girada_exif or 'ninguna'}")
print(f"Sin origen o licencia: {len(sin_origen)}/{len(MANIFIESTO)}")
MANIFIESTO[["megapixeles", "ancho", "alto"]].describe().round(2).T

## 3. Caché de percepción

Para cada imagen se ejecuta **el mismo código que el sistema** (`_cache_percepcion.py`, derivado de la
celda 4 del NB02) en un proceso aparte, para que este kernel no cargue YOLO:

1. `SharedPerceptionEngine.detect()`: `bbox`, `fuente`, `clase`, `sujeto_discreto`…;
2. **todas** las cajas de YOLO, no solo la elegida;
3. la máscara del sujeto con la misma bifurcación que `EspacioAislamientoEngine.analizar`.

La caché guarda el MD5 de cada imagen, las constantes de percepción y segmentación y el MD5 de los pesos
de YOLO. Si algo cambia, `comun.vigencia_cache_percepcion()` la da por obsoleta y se reconstruye sola.
Así se evita lo que pasó con `imagen61`. La primera vez tarda unos 8 minutos.

In [ ]:
RECALCULAR_CACHE = False

comun.construir_cache_percepcion(RECALCULAR_CACHE)
META, PERC, MASK = comun.cargar_cache_percepcion()

P = pd.DataFrame.from_dict({s: {
    "fuente": d["fuente"], "clase": d["clase"] or "", "confianza_yolo": d["confianza_yolo"],
    "n_cajas": len(d["cajas"]), "fuente_mascara": d["fuente_mascara"],
    "cobertura_mascara": d["cobertura_mascara"], "blob_area_ratio": d["blob_area_ratio"],
} for s, d in PERC.items()}, orient="index").reindex(STEMS)
YOLO = P.index[P.fuente == "yolo"].tolist()

print(f"Caché del {META['fecha']} (commit {META['commit'][:7]}, {META['segundos']:.0f} s)")
fuentes = P.fuente.value_counts().reindex(["yolo", "saliencia", "sin_sujeto_claro"], fill_value=0)
print("Fuente de la percepción:", fuentes.to_dict(), "· NB02 (antes de sustituir imagen61): 61 / 39 / 0")

# Máscaras degeneradas: casi vacías o casi todo el encuadre (candidatas a falso positivo en E2).
degeneradas = P[(P.cobertura_mascara < 0.01) | (P.cobertura_mascara > 0.95)]
display(degeneradas[["fuente", "clase", "cobertura_mascara"]])
P.loc[YOLO, "clase"].value_counts().rename("imágenes con fuente = yolo").to_frame().T

## 4. Anotaciones

La guía está en [`anotaciones/GUIA_ANOTACION.md`](anotaciones/GUIA_ANOTACION.md). Es un **borrador**:
fíjala antes de anotar.

### 4.1 Plantillas y migración

Se crean solo si no existen:

- **`A1_contexto.csv`**: `imagen, contexto, notas`.
- **`A2_aplicabilidad.csv`**: las columnas de la guía. Se migran las revisiones de
  `notebooks/revision_espacio_aislamiento.csv` (`segmentacion`, `protagonista` y `notas`), con la fuente y la
  clase que se vieron al revisar (`fuente_revisada`, `clase_revisada`). La fila de `imagen61` se vacía: la
  imagen se sustituyó el 2026-09-26 y la revisión era de la duplicada.
- **`patron_percibido.csv`**: se migra de `revision_composicion_espacial.csv`, también con la fila 61 vacía.

Las columnas `horizonte_real` y `perspectiva_real` de `revision_lineas_direccion.csv` están vacías (0/100):
no hay nada que migrar.

In [ ]:
FECHA_SUSTITUCION_61 = "2026-09-26"
NOTA_61 = (f"revisar: la imagen se sustituyó el {FECHA_SUSTITUCION_61} y la revisión anterior era de la "
           "imagen duplicada")
ANOT.mkdir(parents=True, exist_ok=True)
creadas = []


def crear_si_falta(nombre, df):
    ruta = ANOT / f"{nombre}.csv"
    if ruta.exists():
        return
    df.to_csv(ruta, index=False, encoding="utf-8")
    creadas.append(nombre)


def revision_antigua(nombre):
    """Lee un CSV de notebooks/ indexado por `imagenN`, o None si no existe."""
    ruta = RAIZ / "notebooks" / f"{nombre}.csv"
    if not ruta.exists():
        return None
    rev = comun.leer_csv_robusto(ruta)
    return rev.assign(imagen="imagen" + rev["n"].str.strip()).set_index("imagen").reindex(STEMS).fillna("")


# A1
crear_si_falta("A1_contexto", pd.DataFrame({"imagen": STEMS, "contexto": "", "notas": ""}))

# A2
a2 = pd.DataFrame("", index=pd.Index(STEMS, name="imagen"), columns=[*comun.ESQUEMA_A2, "fuente_revisada",
                                                                     "clase_revisada", "notas"])
a2["fuente_revisada"], a2["clase_revisada"] = P.fuente, P.clase
rev = revision_antigua("revision_espacio_aislamiento")
if rev is not None:
    for col in ("segmentacion", "protagonista"):
        rev[col] = rev[col].map(comun.normalizar_valor)
    revisadas = rev.index[(rev.segmentacion != "") | (rev.protagonista != "")]
    a2.loc[revisadas, ["segmentacion", "protagonista"]] = rev.loc[revisadas, ["segmentacion", "protagonista"]].values
    a2.loc[revisadas, "fuente_revisada"] = rev.loc[revisadas, "fuente"].values
    a2.loc[revisadas, "clase_revisada"] = rev.loc[revisadas, "clase"].values
    a2["notas"] = rev["notas"].str.strip()
    a2.loc["imagen61", ["segmentacion", "protagonista"]] = ""
    a2.loc["imagen61", ["fuente_revisada", "clase_revisada"]] = P.loc["imagen61", ["fuente", "clase"]].values
    a2.loc["imagen61", "notas"] = f"{NOTA_61}. {a2.at['imagen61', 'notas']}".strip(". ")
crear_si_falta("A2_aplicabilidad", a2.reset_index())

# Patrón percibido (análisis complementario de §9.5)
pat = pd.DataFrame({"imagen": STEMS, "patron_percibido": "", "notas": ""}).set_index("imagen")
rev = revision_antigua("revision_composicion_espacial")
if rev is not None:
    pat["patron_percibido"] = rev["patron_percibido"].map(comun.normalizar_valor)
    pat["notas"] = rev["notas"].str.strip()
    pat.loc["imagen61", "patron_percibido"] = ""
    pat.loc["imagen61", "notas"] = f"{NOTA_61}. {pat.at['imagen61', 'notas']}".strip(". ")
crear_si_falta("patron_percibido", pat.reset_index())

print("Plantillas creadas ahora:", creadas or "ninguna (ya existían: no se tocan)")

### 4.2 Vigencia de las revisiones de segmentación

Una revisión de `segmentacion` o `protagonista` evalúa una caja y una máscara concretas. Si la percepción
actual tiene otra fuente u otra clase de YOLO que la revisada, la máscara ya no es la misma y hay que
revisar la fila otra vez. Tras revisarla, actualiza `fuente_revisada` y `clase_revisada` con los valores
actuales, que salen en el título de la hoja de segmentación.

In [ ]:
A1 = comun.leer_anotacion("A1_contexto", comun.ESQUEMA_A1)
A2 = comun.leer_anotacion("A2_aplicabilidad", comun.ESQUEMA_A2)
PAT = comun.leer_anotacion("patron_percibido", comun.ESQUEMA_PATRON)
for nombre, df in {"A1": A1, "A2": A2, "patron_percibido": PAT}.items():
    assert sorted(df.index, key=lambda s: int(s[6:])) == STEMS, f"{nombre}: las filas no son imagen1..imagen100"

con_revision = A2.index[(A2.segmentacion != "") | (A2.protagonista != "")]
OBSOLETAS = comun.revisiones_obsoletas(A2, PERC)
print(f"Revisiones de segmentación: {len(con_revision)} · obsoletas (hay que revisarlas de nuevo): {len(OBSOLETAS)}")
OBSOLETAS

### 4.3 Hojas de revisión

Doce imágenes por página:

- **`ciega_NN.png`**: solo la foto, orientada como la ven los especialistas. Sirve para A1 y las columnas
  ciegas de A2; rellénalas **antes** de ver las de segmentación.
- **`segmentacion_NN.png`**: la máscara del sujeto (cian) y la caja de GrabCut (roja) en el espacio de
  trabajo, primero las de `fuente = yolo`. El título lleva la fuente, la clase de YOLO y su confianza, y la
  cobertura de la máscara.

Se regeneran si cambia la caché. Aquí se muestra solo la primera página de cada tipo; el resto está en
`outputs/experimentos/hojas_revision/`.

In [ ]:
REGENERAR_HOJAS = False
POR_HOJA, COLUMNAS = 12, 4
CIAN, ROJO = np.array([0, 229, 255]), (230, 40, 40)


def miniatura(s, lado=900):
    with Image.open(RUTA[s]) as im:
        im.draft("RGB", (lado, lado))
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((lado, lado))
        return np.asarray(im)


def con_mascara(s):
    rgb = cv2.cvtColor(ea.reescalar(cargar_imagen(RUTA[s])), cv2.COLOR_BGR2RGB)
    m = MASK[s]
    rgb[m] = (0.45 * rgb[m] + 0.55 * CIAN).astype(np.uint8)
    contornos, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(rgb, contornos, -1, CIAN.tolist(), 2)
    if PERC[s]["rect"]:
        x, y, w, h = PERC[s]["rect"]
        cv2.rectangle(rgb, (x, y), (x + w, y + h), ROJO, 3)
    return rgb


def titulo_segmentacion(s):
    r = P.loc[s]
    yolo = f" · {r.clase} {r.confianza_yolo:.2f}" if r.fuente == "yolo" else ""
    return f"{s} · {r.fuente}{yolo}\ncobertura máscara {r.cobertura_mascara:.2f}"


def paginas(stems, dibujar, titulo, prefijo):
    rutas = []
    for k in range(0, len(stems), POR_HOJA):
        grupo = stems[k:k + POR_HOJA]
        filas = int(np.ceil(len(grupo) / COLUMNAS))
        fig, axs = plt.subplots(filas, COLUMNAS, figsize=(COLUMNAS * 5, filas * 4.3))
        for ax in np.atleast_1d(axs).ravel():
            ax.axis("off")
        for ax, s in zip(np.atleast_1d(axs).ravel(), grupo):
            ax.imshow(dibujar(s))
            ax.set_title(titulo(s), fontsize=11, loc="left")
        fig.tight_layout()
        ruta = HOJAS / f"{prefijo}_{k // POR_HOJA + 1:02d}.png"
        fig.savefig(ruta, dpi=110)
        plt.close(fig)
        rutas.append(ruta)
    return rutas


sello = HOJAS / "_cache.txt"
if REGENERAR_HOJAS or not sello.exists() or sello.read_text(encoding="utf-8") != META["fecha"]:
    HOJAS.mkdir(parents=True, exist_ok=True)
    for vieja in HOJAS.glob("*.png"):
        vieja.unlink()
    orden_seg = YOLO + [s for s in STEMS if s not in YOLO]
    n_ciega = len(paginas(STEMS, miniatura, lambda s: s, "ciega"))
    n_seg = len(paginas(orden_seg, con_mascara, titulo_segmentacion, "segmentacion"))
    sello.write_text(META["fecha"], encoding="utf-8")
    print(f"Hojas generadas: {n_ciega} ciegas y {n_seg} de segmentación en {comun.relativa(HOJAS)}")
else:
    print(f"Hojas vigentes en {comun.relativa(HOJAS)} (REGENERAR_HOJAS = True para rehacerlas)")

display(ImagenIPy(filename=str(HOJAS / "ciega_01.png"), width=900))
display(ImagenIPy(filename=str(HOJAS / "segmentacion_01.png"), width=900))

### 4.4 Validación de las anotaciones

Por columna se comprueba si está completa (todas las filas que deben anotarse lo están) y si todos los
valores son válidos. `segmentacion` y `protagonista` son obligatorias solo en las imágenes con
`fuente = yolo` en la percepción actual. A2 no cuenta como completa mientras haya revisiones obsoletas
(§4.2).

In [ ]:
V = pd.concat({
    "A1": comun.validar_anotacion(A1, comun.ESQUEMA_A1),
    "A2": comun.validar_anotacion(A2, comun.ESQUEMA_A2, {c: YOLO for c in comun.A2_SOLO_YOLO}),
    "patron_percibido": comun.validar_anotacion(PAT, comun.ESQUEMA_PATRON),
})
invalidas = {k: v for k, v in V.invalidas.items() if v}
assert not invalidas, f"Valores fuera del esquema (corrígelos en el CSV): {invalidas}"

A1_COMPLETA = bool(V.loc["A1"].completa.all())
A2_COMPLETA = bool(V.loc["A2"].completa.all()) and OBSOLETAS.empty
print(f"A1 completa: {A1_COMPLETA} · A2 completa: {A2_COMPLETA} (revisiones obsoletas: {len(OBSOLETAS)})")
V[["requeridas", "anotadas", "completa"]]

## 5. Tablas descriptivas de §8.3

- **Tabla de conjuntos:** C1, O, val, C2 y R, con su tamaño, origen y uso (la tanda V20 se retiró en la versión
  acotada, experimentos.md §15).
- **Composición de C2:** resolución, orientación, fuente de la percepción, reparto por contexto (A1) y casos
  límite (A2).

Mientras falten anotaciones, las filas que dependen de ellas indican cuántas imágenes están anotadas.

In [ ]:
SPLITS = comun.ruta_config("eva_proyecto") / "data" / "splits"
n_split = {k: len(pd.read_csv(SPLITS / f"{f}.csv")) for k, f in [("C1", "test"), ("O", "otro"), ("val", "val")]}
listas = CFG.get("listas") or {}


def n_lista(nombre):
    return len(listas[nombre]) if listas.get(nombre) else "pendiente"


CONJUNTOS = pd.DataFrame([
    ("C1", "Prueba de EVA (test.csv)", n_split["C1"], "E1a, E1b"),
    ("O", "other de EVA (otro.csv, sort = 6), near-OOD", n_split["O"], "E1b, E1c"),
    ("val", "Validación de EVA (val.csv)", n_split["val"], "E1c (control del umbral)"),
    ("C2", "Corpus de desarrollo (data/)", len(C2), "E1d, E2, E3, E4b"),
    ("R", "Subconjunto de C2 × 3 repeticiones", n_lista("R"), "E4c"),
], columns=["conjunto", "descripción", "n", "experimentos"]).set_index("conjunto")
CONJUNTOS.to_csv(RES / "tabla_conjuntos.csv", encoding="utf-8")
display(CONJUNTOS)


def recuento(serie, n_anotadas, etiqueta):
    t = serie.value_counts().rename("n").to_frame()
    t["%"] = (100 * t.n / len(C2)).round(1)
    t.index = [f"{etiqueta}: {i}" for i in t.index]
    return t


mp = MANIFIESTO.megapixeles
bloques = [
    pd.DataFrame({"n": [len(C2)], "%": [100.0]}, index=["imágenes"]),
    pd.DataFrame({"n": [f"{mp.min():.2f} / {mp.median():.2f} / {mp.max():.2f}"], "%": [""]},
                 index=["megapíxeles (mín / mediana / máx)"]),
    recuento(MANIFIESTO.orientacion, len(C2), "orientación"),
    recuento(P.fuente, len(C2), "fuente de la percepción"),
]
n_a1 = int((A1.contexto != "").sum())
bloques.append(recuento(A1.contexto[A1.contexto != ""], n_a1, f"contexto A1 ({n_a1}/100 anotadas)"))
casos_limite = {   # caso límite -> columna de A2 y valor que lo define
    "B/N o sin color (color_analizable = no)": ("color_analizable", "no"),
    "sin sujeto (sin_sujeto = si)": ("sin_sujeto", "si"),
    "horizonte natural": ("horizonte_natural", "si"),
    "perspectiva": ("perspectiva", "si"),
}
bloques.append(pd.DataFrame(
    {"n": [int((A2[c] == v).sum()) for c, v in casos_limite.values()],
     "%": [round(100 * (A2[c] == v).mean(), 1) for c, v in casos_limite.values()]},
    index=[f"caso límite: {k} ({int((A2[c] != '').sum())}/100 anotadas)" for k, (c, _) in casos_limite.items()]))
COMPOSICION = pd.concat(bloques)
COMPOSICION.to_csv(RES / "tabla_composicion_C2.csv", encoding="utf-8")
COMPOSICION

## 6. Propuesta de R

Selección **estratificada y reproducible** (semilla de la configuración), hecha solo con A1 y la percepción,
sin mirar resultados del sistema:

- por cada uno de los 5 contextos de A1, `R_por_contexto` imágenes (1 en la versión acotada: R = 5);
- dentro de cada contexto se alternan `fuente = yolo` (gates de identidad abiertos) y `saliencia`
  (cerrados), para que las dos ramas estén representadas;
- las imágenes con A1 = `otro` no entran: no tienen un vector W de contexto propio.

El notebook no escribe la configuración: imprime el bloque para pegarlo en `config_experimentos.yaml`
(`listas:`) y lo deja a tu decisión. Una vez congelado, no se cambia.

In [ ]:
def proponer(n_por_contexto, semilla):
    rng = np.random.default_rng(semilla)
    seleccion = []
    for ctx in comun.ETIQUETAS_SISTEMA:
        candidatas = [s for s in STEMS if A1.at[s, "contexto"] == ctx]
        grupos = [rng.permutation([s for s in candidatas if s in YOLO]).tolist(),
                  rng.permutation([s for s in candidatas if s not in YOLO]).tolist()]
        # Con cuota 1 la alternancia empezaría siempre por yolo: el primer grupo se sortea.
        if rng.random() < 0.5:
            grupos.reverse()
        elegidas = []
        while len(elegidas) < n_por_contexto and any(grupos):
            for g in grupos:
                if g and len(elegidas) < n_por_contexto:
                    elegidas.append(g.pop(0))
        if len(elegidas) < n_por_contexto:
            print(f"Aviso: {ctx} solo tiene {len(elegidas)} imágenes para una cuota de {n_por_contexto}")
        seleccion += sorted(elegidas, key=lambda s: int(s[6:]))
    return seleccion


if A1_COMPLETA:
    propuesta = {"R": proponer(CFG["seleccion"]["R_por_contexto"], CFG["semilla"])}
    for nombre, lista in propuesta.items():
        resumen = pd.crosstab(A1.loc[lista, "contexto"], P.loc[lista, "fuente"], margins=True, margins_name="total")
        display(Markdown(f"**{nombre}** ({len(lista)} imágenes)")); display(resumen)
    print("Bloque para config_experimentos.yaml:\n")
    print("listas:\n" + "".join(f"  {k}: [{', '.join(v)}]\n" for k, v in propuesta.items()))
    if listas.get("R"):
        print("Lista ya fijada en la configuración:", listas["R"])
else:
    print(f"Pendiente: la propuesta necesita A1 completa ({n_a1}/100 anotadas).")

## 7. Congelación

Fija en `experimentos/congelacion.json` todo lo que decide los resultados:

- el commit del código;
- el MD5 de las imágenes de C2, del manifiesto, de los modelos (clasificador y YOLO), de W y del modelo nulo,
  de los prompts (`agents.yaml`, `tasks.yaml`), de la configuración de experimentos y de las anotaciones;
- las constantes de `tools/*.py` y `crew.py` (leídas del código fuente), y los modelos LLM;
- las versiones de los paquetes, la GPU y CUDA;
- los criterios de éxito, la lista R y la tabla de precios de la API.

Después, cada notebook E* llama a `comun.comprobar_congelacion()` y avisa de cualquier diferencia. Commitear
`congelacion.json` cambia HEAD, pero solo cuenta como diferencia si cambia código en `src/` o `models/`.

**Bloqueos**, que impiden congelar: árbol con cambios sin commit, caché obsoleta, manifiesto desfasado o
congelación ya existente. Para recongelar, borra el fichero a mano: queda en el historial de git.

**Pendientes**, que impiden congelar salvo con `FORZAR = True` (se registra en el JSON): anotaciones
incompletas, guía sin fijar, criterios a `null`, lista R vacía o precios sin fijar.

In [ ]:
CONGELAR = False
FORZAR = False
RUTA_CONGELACION = RAIZ / "experimentos" / "congelacion.json"

bloqueos, pendientes = [], []
if comun._git("status", "--porcelain"):
    bloqueos.append("hay cambios sin commit (git status)")
if comun.vigencia_cache_percepcion() is not None:
    bloqueos.append(f"caché de percepción obsoleta: {comun.vigencia_cache_percepcion()}")
if any(MANIFIESTO.at[s, "md5"] != comun.md5(RUTA[s]) for s in STEMS):
    bloqueos.append("el manifiesto no coincide con data/ (vuelve a ejecutar §2)")
if RUTA_CONGELACION.exists():
    bloqueos.append("ya existe congelacion.json (bórralo a mano para recongelar)")

if not A1_COMPLETA:
    pendientes.append("A1 incompleta")
if not A2_COMPLETA:
    pendientes.append("A2 incompleta o con revisiones obsoletas")
if "_pendiente_" in (ANOT / "GUIA_ANOTACION.md").read_text(encoding="utf-8"):
    pendientes.append("la guía de anotación no tiene fecha de fijación")
if sin_origen:
    pendientes.append(f"{len(sin_origen)} imágenes sin origen o licencia en el manifiesto")
CRITERIOS = {k: v["criterios"] for k, v in CFG.items() if isinstance(v, dict) and "criterios" in v}
nulos = [f"{e}.{c}" for e, cs in CRITERIOS.items() for c, v in cs.items() if v is None]
if nulos:
    pendientes.append("criterios sin fijar: " + ", ".join(nulos))
if not listas.get("R"):
    pendientes.append("lista R sin fijar")
PRECIOS = CFG.get("precios") or {}
if PRECIOS.get("fecha_consulta") is None or PRECIOS.get("usd_por_eur") is None or any(
        v is None for m in (PRECIOS.get("modelos") or {}).values() for v in m.values()):
    pendientes.append("precios de la API sin fijar (precios:)")

display(Markdown("**Bloqueos:** " + ("; ".join(bloqueos) or "ninguno") +
                 "  \n**Pendientes:** " + ("; ".join(pendientes) or "ninguno")))

if CONGELAR:
    assert not bloqueos, f"No se puede congelar: {bloqueos}"
    assert FORZAR or not pendientes, f"Hay pendientes (FORZAR = True para congelar igualmente): {pendientes}"
    proc = comun.procedencia()
    congelacion = {
        **comun.estado_sistema(),
        "fecha": proc["fecha"], "python": proc["python"], "plataforma": proc["plataforma"],
        "gpu": proc["gpu"], "cuda": proc["cuda"], "versiones": proc["versiones"],
        "criterios": CRITERIOS, "listas": listas, "precios": PRECIOS, "cache_percepcion": META,
        "forzada": FORZAR, "pendientes_al_congelar": pendientes,
    }
    comun.guardar_json(congelacion, RUTA_CONGELACION)
    print(f"Congelado en {comun.relativa(RUTA_CONGELACION)} con el commit {congelacion['commit'][:7]}. "
          "Haz commit de ese fichero antes de lanzar las tandas.")
else:
    print("No se congela (CONGELAR = False).")